# Customer Churn Prediction

## Predicting Customers at Risk of Churning Using Machine Learning

This project develops a machine learning classification system to predict whether a telecommunications customer is likely to churn.

The workflow covers exploratory data analysis, preprocessing, model comparison, hyperparameter tuning, classification threshold optimization, and explainable AI using SHAP.

## 1. Business Problem

Customer churn is a major challenge for telecommunications companies because losing existing customers can negatively affect revenue and increase acquisition costs.

### Objective

Build a binary classification model that estimates the probability of customer churn and helps prioritize customers for retention campaigns. The project also aims to explain the factors behind model predictions.

## 2. Dataset and Initial Exploration

The project uses the Telco Customer Churn dataset containing 7,043 customers and 21 original columns. The data includes demographic information, subscribed services, contract details, billing information, and the target variable `Churn`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/content/Telco-Customer-Churn.CSV")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
df.info()

### Data Quality

`TotalCharges` contains blank strings that are not initially detected as standard missing values. These values are converted to numeric values with `errors="coerce"` and then handled as missing values. For customers with zero tenure and blank total charges, replacing the missing total with 0 is consistent with the billing interpretation.

In [ ]:
print("Blank TotalCharges before conversion:", (df["TotalCharges"].astype(str).str.strip() == "").sum())

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing TotalCharges after conversion:", df["TotalCharges"].isna().sum())

df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("Missing values after handling TotalCharges:", df.isna().sum().sum())

In [ ]:
print("Churn counts:")
print(df["Churn"].value_counts())

print("\nChurn percentage:")
print(df["Churn"].value_counts(normalize=True).mul(100))

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nUnique values per column:")
print(df.nunique())

## 3. Exploratory Data Analysis

The exploratory analysis examines the distribution of churn and investigates relationships between churn and contract type, tenure, monthly charges, internet service, payment method, and additional services.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
contract_churn

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Customer Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.show()

### Insight — Contract Type

Month-to-month customers have a substantially higher churn rate than customers with one-year or two-year contracts. This makes contract type an important retention signal.

In [ ]:
tenure_summary = df.groupby("Churn")["tenure"].describe()
tenure_summary

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, kde=True)
plt.title("Customer Tenure Distribution by Churn")
plt.xlabel("Tenure (Months)")
plt.ylabel("Number of Customers")
plt.show()

### Insight — Tenure

Customers who churned had substantially shorter tenure than customers who stayed. The median tenure was 10 months for churned customers compared with 38 months for customers who remained.

In [ ]:
monthly_charges_summary = df.groupby("Churn")["MonthlyCharges"].describe()
monthly_charges_summary

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

### Insight — Monthly Charges

Customers who churned had higher monthly charges on average than customers who remained. This suggests that pricing or the combination of services associated with higher monthly charges may be relevant to churn risk.

In [ ]:
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
internet_churn

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="InternetService", hue="Churn")
plt.title("Customer Churn by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Number of Customers")
plt.show()

### Insight — Internet Service

Fiber-optic customers show a substantially higher churn rate than DSL customers and customers without internet service.

In [ ]:
service_columns = [
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies"
]

service_churn = {}
for col in service_columns:
    service_churn[col] = pd.crosstab(
        df[col], df["Churn"], normalize="index"
    ).mul(100)

service_churn

In [ ]:
payment_churn = pd.crosstab(
    df["PaymentMethod"], df["Churn"], normalize="index"
) * 100
payment_churn

In [ ]:
payment_churn.plot(kind="bar", stacked=True, figsize=(9, 5))
plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Percentage")
plt.legend(title="Churn")
plt.xticks(rotation=20)
plt.show()

### Insight — Payment Method

Customers using electronic check have the highest observed churn rate among the payment methods.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="OnlineSecurity", hue="Churn")
plt.title("Customer Churn by Online Security")
plt.xlabel("Online Security")
plt.ylabel("Number of Customers")
plt.show()

## 4. Data Preprocessing

The `customerID` column is an identifier and is removed because it does not provide meaningful predictive information. The target is encoded as 0 = No Churn and 1 = Churn.

The data is split using an 80/20 stratified train-test split. Categorical variables are one-hot encoded. Numerical variables are standardized for models that benefit from scaling, while tree-based models use the numerical values without scaling. Preprocessing is implemented inside pipelines to prevent data leakage.

In [ ]:
df = df.drop("customerID", axis=1)
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))
print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Categorical features:")
print(categorical_features)
print("\nNumerical features:")
print(numerical_features)

## 5. Model Training and Comparison

Six classification algorithms are evaluated:

1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Tree
4. Random Forest
5. XGBoost
6. LightGBM

Accuracy, Precision, Recall, F1 Score, and ROC-AUC are used for evaluation. ROC-AUC is the primary comparison metric, while Recall and F1 are also important because identifying customers at risk of churn is the main business objective.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

logistic_model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(max_iter=1000))
])

knn_model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

tree_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", DecisionTreeClassifier(random_state=42, max_depth=5))
])

rf_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
])

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

xgb_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        random_state=42, eval_metric="logloss"
    ))
])

lgbm_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model", LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        random_state=42, verbosity=-1
    ))
])

In [ ]:
models = {
    "Logistic Regression": logistic_model,
    "KNN": knn_model,
    "Decision Tree": tree_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model
}

model_results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = pred
    probabilities[name] = prob
    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results = pd.DataFrame(model_results).sort_values("ROC-AUC", ascending=False)
results

In [ ]:
plt.figure(figsize=(9, 5))
results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(
    kind="bar", figsize=(10, 5)
)
plt.title("Initial Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=25)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

### Initial Comparison

Logistic Regression achieved the strongest initial overall performance, while XGBoost and LightGBM were close competitors. The three strongest candidates were selected for hyperparameter tuning.

## 6. Hyperparameter Tuning

RandomizedSearchCV with 5-fold cross-validation is used to tune Logistic Regression, XGBoost, and LightGBM. The test set remains untouched during hyperparameter selection.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

logistic_params = {
    "model__C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "model__solver": ["liblinear", "lbfgs"],
    "model__class_weight": [None, "balanced"]
}

logistic_search = RandomizedSearchCV(
    logistic_model, logistic_params, n_iter=10, scoring="roc_auc",
    cv=5, random_state=42, n_jobs=-1
)
logistic_search.fit(X_train, y_train)

print("Logistic Regression best parameters:")
print(logistic_search.best_params_)
print("Best CV ROC-AUC:", logistic_search.best_score_)

best_logistic = logistic_search.best_estimator_
y_pred_log = best_logistic.predict(X_test)
y_prob_log = best_logistic.predict_proba(X_test)[:, 1]

In [ ]:
xgb_params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_model, xgb_params, n_iter=20, scoring="roc_auc",
    cv=5, random_state=42, n_jobs=-1
)
xgb_search.fit(X_train, y_train)

print("XGBoost best parameters:")
print(xgb_search.best_params_)
print("Best CV ROC-AUC:", xgb_search.best_score_)

best_xgb = xgb_search.best_estimator_
y_pred_xgb_tuned = best_xgb.predict(X_test)
y_prob_xgb_tuned = best_xgb.predict_proba(X_test)[:, 1]

In [ ]:
lgbm_params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [-1, 2, 3, 4, 5],
    "model__num_leaves": [7, 15, 31, 50],
    "model__min_child_samples": [10, 20, 30, 50],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0]
}

lgbm_search = RandomizedSearchCV(
    lgbm_model, lgbm_params, n_iter=20, scoring="roc_auc",
    cv=5, random_state=42, n_jobs=-1
)
lgbm_search.fit(X_train, y_train)

print("LightGBM best parameters:")
print(lgbm_search.best_params_)
print("Best CV ROC-AUC:", lgbm_search.best_score_)

best_lgbm = lgbm_search.best_estimator_
y_pred_lgbm_tuned = best_lgbm.predict(X_test)
y_prob_lgbm_tuned = best_lgbm.predict_proba(X_test)[:, 1]

In [ ]:
tuned_results = pd.DataFrame({
    "Model": ["Logistic Regression (Tuned)", "XGBoost (Tuned)", "LightGBM (Tuned)"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_xgb_tuned),
        accuracy_score(y_test, y_pred_lgbm_tuned)
    ],
    "Precision": [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_xgb_tuned),
        precision_score(y_test, y_pred_lgbm_tuned)
    ],
    "Recall": [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_xgb_tuned),
        recall_score(y_test, y_pred_lgbm_tuned)
    ],
    "F1": [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_xgb_tuned),
        f1_score(y_test, y_pred_lgbm_tuned)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_log),
        roc_auc_score(y_test, y_prob_xgb_tuned),
        roc_auc_score(y_test, y_prob_lgbm_tuned)
    ]
}).sort_values("ROC-AUC", ascending=False)
tuned_results

## 7. Classification Threshold Optimization

The default threshold of 0.50 is not necessarily optimal for a churn-retention problem. Missing a customer who actually churns can be costly, so different thresholds are evaluated using cross-validation predictions on the training data.

The threshold is selected without using the test set.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_cv_prob = cross_val_predict(
    xgb_search.best_estimator_, X_train, y_train, cv=cv,
    method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.arange(0.30, 0.61, 0.05)
threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (xgb_cv_prob >= threshold).astype(int)
    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, y_pred_threshold),
        "Recall": recall_score(y_train, y_pred_threshold),
        "F1": f1_score(y_train, y_pred_threshold)
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], marker="o", label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker="o", label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["F1"], marker="o", label="F1")
plt.axvline(0.35, linestyle="--", label="Selected Threshold = 0.35")
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Threshold Optimization")
plt.legend()
plt.show()

### Threshold Selection

A threshold of **0.35** was selected because it produced the highest F1 Score among the evaluated thresholds while substantially improving Recall. This choice prioritizes identifying more potential churners for retention campaigns.

In [ ]:
final_threshold = 0.35
y_pred_final = (y_prob_xgb_tuned >= final_threshold).astype(int)

## 8. Final Model Evaluation

The final model is the tuned XGBoost classifier. The selected threshold of 0.35 is applied to its predicted probabilities.

In [ ]:
final_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_final),
    "Precision": precision_score(y_test, y_pred_final),
    "Recall": recall_score(y_test, y_pred_final),
    "F1 Score": f1_score(y_test, y_pred_final),
    "ROC-AUC": roc_auc_score(y_test, y_prob_xgb_tuned)
}

pd.Series(final_metrics, name="Score")

In [ ]:
cm_final = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(cm_final, display_labels=["No Churn", "Churn"])
disp.plot()
plt.title("Final XGBoost Confusion Matrix (Threshold = 0.35)")
plt.show()

### Final Test Performance

The selected threshold increases the model's sensitivity to churners. On the unseen test set, the final model achieved approximately **73.8% Recall**, **64.4% F1 Score**, and **84.7% ROC-AUC**.

In [ ]:
fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb_tuned)
fpr_lgbm, tpr_lgbm, _ = roc_curve(y_test, y_prob_lgbm_tuned)

auc_log = auc(fpr_log, tpr_log)
auc_xgb = auc(fpr_xgb, tpr_xgb)
auc_lgbm = auc(fpr_lgbm, tpr_lgbm)

plt.figure(figsize=(8, 6))
plt.plot(fpr_log, tpr_log, label=f"Logistic Regression (AUC = {auc_log:.3f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {auc_xgb:.3f})")
plt.plot(fpr_lgbm, tpr_lgbm, label=f"LightGBM (AUC = {auc_lgbm:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_prob_xgb_tuned)
average_precision = average_precision_score(y_test, y_prob_xgb_tuned)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f"XGBoost (AP = {average_precision:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - XGBoost")
plt.legend()
plt.show()

## 9. Model Explainability with SHAP

SHAP (SHapley Additive exPlanations) is used to explain the tuned XGBoost model. Global SHAP identifies the most influential features across customers, while local SHAP explains an individual prediction.

In [ ]:
import shap

explainer = shap.TreeExplainer(best_xgb.named_steps["model"])
preprocessor = best_xgb.named_steps["preprocessor"]

X_test_transformed = preprocessor.transform(X_test)
X_test_shap = X_test_transformed
feature_names = preprocessor.get_feature_names_out()
shap_values = explainer.shap_values(X_test_shap)

print("Number of transformed features:", len(feature_names))

In [ ]:
shap.summary_plot(
    shap_values, X_test_shap, feature_names=feature_names, plot_type="bar"
)

### Global SHAP Insights

The most influential features include contract type, tenure, internet service, monthly charges, payment method, online security, and technical support. These results are broadly consistent with the patterns identified during EDA.

In [ ]:
shap.summary_plot(
    shap_values, X_test_shap, feature_names=feature_names
)

In [ ]:
test_probabilities = pd.Series(y_prob_xgb_tuned, index=X_test.index)
highest_risk_customer = test_probabilities.idxmax()

print("Customer index:", highest_risk_customer)
print("Churn probability:", test_probabilities.loc[highest_risk_customer])
print("\nCustomer information:")
print(X_test.loc[highest_risk_customer])

In [ ]:
customer_position = X_test.index.get_loc(highest_risk_customer)
customer_shap_values = shap_values[customer_position]

shap.waterfall_plot(
    shap.Explanation(
        values=customer_shap_values,
        base_values=explainer.expected_value,
        data=X_test_shap[customer_position],
        feature_names=feature_names
    ),
    max_display=15
)

### Example of a High-Risk Customer

The selected customer received a churn probability of approximately **91.6%**. The strongest factors increasing the prediction included very short tenure, a month-to-month contract, fiber-optic internet service, high monthly charges, electronic-check payment, no online security, and no technical support.

## 10. Business Insights and Recommendations

### Key Findings

- Month-to-month customers have substantially higher churn rates than customers with longer contracts.
- Customers with shorter tenure are more likely to churn.
- Fiber-optic customers show higher churn rates than other internet-service groups.
- Higher monthly charges are associated with increased churn.
- Customers without Online Security or Technical Support show higher churn rates.
- Electronic-check users have a notably higher churn rate.

### Recommendations

- Develop an early-retention program for customers during their first months of service.
- Offer incentives for month-to-month customers to switch to longer-term contracts.
- Investigate satisfaction, pricing, and service quality among fiber-optic customers.
- Promote Online Security and Technical Support as part of retention packages.
- Prioritize high-risk customers identified by the model for targeted retention campaigns.

## 11. Conclusion

This project developed a complete customer churn prediction workflow, from exploratory analysis to machine learning and explainability. Six classification algorithms were compared, and the strongest candidates were further optimized using cross-validation and hyperparameter tuning.

The final tuned XGBoost model achieved a **ROC-AUC of 84.70%** on the unseen test set. Reducing the classification threshold from 0.50 to 0.35 increased Recall to **73.80%** and F1 Score to **64.41%**, supporting a retention-focused use case.

SHAP analysis provided interpretable explanations for both global model behavior and individual customer predictions. The resulting system can serve as a decision-support tool for prioritizing customers for retention efforts.